[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bme5014-deeplearning/bme5014-deeplearning/blob/main/F26/recitation_0/Rec-0.2_OOP_Fundamentals_BME5014.ipynb)


# Recitation 0.2: Object-Oriented Programming (OOP) for Biomedical Deep Learning (BME5014)

Welcome back!

**Prerequisite:** Recitation 0.1 (Python Fundamentals) and the ability to run notebooks in Google Colab.

## Why does a deep learning course need OOP?

In deep learning frameworks such as **PyTorch**, almost everything you touch is a **class**: your neural network is a class, your dataset is a class, and even data-preprocessing steps are often classes. You do not need to become a software engineer, but you do need to be able to *read* and *write* these classes comfortably.

**What matters most for this course (focus here first):**
1. Classes and objects, `__init__` and `self`
2. Inheritance and `super()` (every PyTorch model inherits from `nn.Module`)
3. Overriding a method (your model overrides `forward()`)
4. A few "dunder" methods: `__len__`, `__getitem__`, `__call__`

Encapsulation, abstraction, and class attributes are good to know and are covered briefly, but you can return to them later.

> **Coming from MATLAB?** A MATLAB `struct` only *holds data* (`patient.hr = 72`). A Python class *holds data and the functions that work on that data* together, so the data and its behavior travel as one package.


## 1. Why OOP?

Let's start with the approach you already know: separate variables and separate functions.


In [ ]:
# Procedural approach: separate variables + functions for each patient
p1_id, p1_hr, p1_temp = "P-001", 72, 36.6
p2_id, p2_hr, p2_temp = "P-002", 105, 38.4

def has_fever(temp_c):
    return temp_c > 38.0

def summarize(patient_id, hr, temp_c):
    return f"{patient_id}: HR {hr} bpm, temp {temp_c} C, fever={has_fever(temp_c)}"

print(summarize(p1_id, p1_hr, p1_temp))
print(summarize(p2_id, p2_hr, p2_temp))


This works for 2 patients and 3 measurements. Now imagine 1000 patients with 20 measurements each (blood pressure, SpO2, glucose, ...). Every function needs a longer argument list, and it is easy to mix up which values belong to which patient.

A **class** bundles the data of one patient *and* the functions that operate on it into a single object.


## 2. Classes and Objects

- A **class** is a **blueprint** (think: the blank patient intake form).
- An **object** (or **instance**) is **one concrete thing built from that blueprint** (the form filled in for one specific patient).


### In Python, everything is an object!

Even the "primitive" values you used in Recitation 0.1 are instances of classes. This is not trivia: it is the reason the same dot-syntax works on strings, lists, and NumPy arrays (`my_list.append(4)`, `my_array.reshape(3, 3)`).


In [ ]:
import numpy as np

a = "1"
print(type(a))

a = 1
print(type(a))

b = [1, 2, 3]
print(type(b))

c = np.array([1, 2, 3])
print(type(c))


### Let us do it the OOP way!


In [ ]:
class Patient:
    def __init__(self, patient_id: str, heart_rate: int, temp_c: float) -> None:  # constructor
        self.patient_id = patient_id   # instance attribute
        self.heart_rate = heart_rate   # instance attribute
        self.temp_c = temp_c           # instance attribute

    def has_fever(self) -> bool:       # instance method
        return self.temp_c > 38.0

    def describe(self) -> None:
        print(f"{self.patient_id}: HR {self.heart_rate} bpm, temp {self.temp_c} C, fever={self.has_fever()}")


p1 = Patient("P-001", 72, 36.6)    # instantiation
p2 = Patient("P-002", 105, 38.4)

print(p1)                          # not very readable (we fix this in Section 4)
print(type(p1))                    # <class '__main__.Patient'>
print(p1.patient_id, p1.heart_rate)   # access attributes with "."
p1.describe()                      # call a method with "."
p2.describe()


### 2.1 The constructor and `self`

- `__init__` is a **dunder method** ("double underscore"). Python calls it automatically when you create an object.
- `self` refers to **this specific object**. It is how the object keeps track of its own data.

When you write `p1 = Patient("P-001", 72, 36.6)`, Python creates an empty object, hands it to `__init__` as `self`, and the three values you passed fill in `patient_id`, `heart_rate`, and `temp_c`. You never pass `self` yourself; Python does it behind the scenes.

### 2.2 Type hints, docstrings, default values

Type hints and docstrings are not enforced at runtime (just like in Recitation 0.1), but nearly all starter code you will receive uses them. Default argument values let a caller leave out arguments they do not need.


In [ ]:
class Patient:
    def __init__(self, patient_id: str, heart_rate: int, temp_c: float, spo2: float = 98.0) -> None:
        """
        Args:
            patient_id: Unique identifier, e.g. "P-001".
            heart_rate: Heart rate in beats per minute.
            temp_c:     Body temperature in degrees Celsius.
            spo2:       Blood oxygen saturation in percent (default: 98.0).
        """
        self.patient_id = patient_id
        self.heart_rate = heart_rate
        self.temp_c = temp_c
        self.spo2 = spo2

    def needs_attention(self) -> bool:
        """True if the patient has a fever or a low oxygen saturation."""
        return self.temp_c > 38.0 or self.spo2 < 92.0

    def describe(self) -> None:
        """Print a one-line summary."""
        print(f"{self.patient_id}: HR {self.heart_rate} bpm, temp {self.temp_c} C, "
              f"SpO2 {self.spo2}% -> needs attention: {self.needs_attention()}")


p1 = Patient("P-001", 72, 36.6)             # spo2 defaults to 98.0
p2 = Patient("P-002", 105, 38.4, 90.0)

p1.describe()
p2.describe()


## 3. Core Concepts: The Four Pillars

OOP rests on four ideas: **Encapsulation**, **Abstraction**, **Inheritance**, and **Polymorphism**. We build each one with a small biomedical example.

### 3.1 Encapsulation

Two ideas:
1. **Bundle** data and methods inside a class.
2. **Control access.** Hide internal state and expose a clean interface.

Think of an infusion pump: a nurse may change the infusion rate, but only within safe limits. The pump does not let anyone type in any number. That is encapsulation.

Python uses naming conventions rather than hard access modifiers like C++:

| Convention | Meaning |
|---|---|
| `attribute` | Public: accessible anywhere |
| `_attribute` | Protected: internal use (by convention) |
| `__attribute` | Private: name-mangled, actively discouraged from outside |


In [ ]:
class InfusionPump:
    def __init__(self, drug: str, rate_ml_h: float = 0.0) -> None:
        self.drug = drug                 # public
        self._rate_ml_h = rate_ml_h      # protected: managed through methods
        self.__lock_code = 4321          # private: should never be accessed directly

    def get_rate(self) -> float:         # getter
        return self._rate_ml_h

    def set_rate(self, rate_ml_h: float) -> None:    # setter with validation
        if rate_ml_h < 0 or rate_ml_h > 500:
            raise ValueError("Rate must be between 0 and 500 mL/h.")
        self._rate_ml_h = rate_ml_h

    def describe(self) -> None:
        print(f"{self.drug} pump - rate: {self._rate_ml_h:.1f} mL/h")


pump = InfusionPump("Saline", 100.0)

print(pump.drug)            # public: fine
print(pump.get_rate())      # controlled access via getter

pump.set_rate(150.0)        # validated update
pump.describe()

try:
    pump.set_rate(5000.0)   # rejected by validation
except ValueError as e:
    print("Rejected:", e)

try:
    print(pump.__lock_code)   # private: cannot be reached from outside
except AttributeError as e:
    print("AttributeError:", e)


The setter lets you **validate before changing**. Direct access (`pump._rate_ml_h = 99999`) would skip all of that, which is exactly why the underscore convention exists: it signals "do not reach in here directly." In a real medical device, guards like this are a safety feature.


### 3.2 Abstraction

**Abstraction** means hiding complexity behind a clean interface. The user calls simple methods and does not need to know what happens inside.

Think of a patient monitor: you read the numbers on the screen. You never see the sensor filtering, the artifact rejection, or the data buffer behind them.

Internal helpers are prefixed with `_` to signal that they are not part of the public interface.


In [ ]:
class VitalsLog:
    def __init__(self, patient_id: str) -> None:
        self.patient_id = patient_id
        self._readings = []

    # ---- Internal helpers: not for users ----
    def _validate(self, bpm: float) -> None:
        if bpm <= 0:
            raise ValueError("Heart rate must be positive.")

    def _record(self, bpm: float) -> None:
        self._readings.append(bpm)

    # ---- Public interface ----
    def add_reading(self, bpm: float) -> None:
        """Add a heart-rate reading (bpm) to the log."""
        self._validate(bpm)
        if bpm > 300:
            print(f"Rejected {bpm}: not physiologically plausible.")
            return
        self._record(bpm)

    def report(self) -> None:
        """Print a summary of all stored readings."""
        print(f"Vitals report - patient {self.patient_id}")
        print("-" * 42)
        for i, bpm in enumerate(self._readings, 1):
            print(f"{i}. {bpm:6.1f} bpm")
        print("-" * 42)
        average = sum(self._readings) / len(self._readings)
        print(f"Average: {average:.1f} bpm | min: {min(self._readings):.1f} | max: {max(self._readings):.1f}")


log = VitalsLog("P-001")
log.add_reading(72)
log.add_reading(75)
log.add_reading(70)
log.add_reading(999)    # should be rejected gracefully

print()
log.report()


The user only calls `add_reading()` and `report()`. The internal `_validate` and `_record` helpers stay hidden. That is abstraction.

You can also enforce abstraction formally with Python's `abc` module (`@abstractmethod`); see the further-reading links at the end.


### 3.3 Inheritance

**Inheritance** lets a child class **reuse and extend** a parent class.

- **Parent class**: the general blueprint
- **Child class**: more specific; inherits the parent's attributes and methods via `super()`
- Children can **override** parent methods to change behavior.

Here a general biosignal recording gets more specific: an ECG recording *is a* biosignal recording, and a Holter (24-hour) recording *is an* ECG recording.

Keep an eye on `super().__init__(...)` below. It is the single most important line to understand before you touch PyTorch.


In [ ]:
# Parent
class BiosignalRecording:
    def __init__(self, patient_id: str, sampling_rate_hz: float, samples: list) -> None:
        self.patient_id = patient_id
        self.sampling_rate_hz = sampling_rate_hz
        self.samples = samples

    def duration_seconds(self) -> float:
        return len(self.samples) / self.sampling_rate_hz

    def describe(self) -> None:
        print(f"[Signal] {self.patient_id} - {self.duration_seconds():.1f} s at {self.sampling_rate_hz:.0f} Hz")


# Child: adds the ECG lead
class ECGRecording(BiosignalRecording):
    def __init__(self, patient_id: str, sampling_rate_hz: float, samples: list, lead: str = "II") -> None:
        super().__init__(patient_id, sampling_rate_hz, samples)   # inherit parent attributes
        self.lead = lead                                          # new attribute

    def peak_amplitude(self) -> float:                            # new method
        return max(self.samples)

    def describe(self) -> None:                                   # override
        print(f"[ECG lead {self.lead}] {self.patient_id} - {self.duration_seconds():.1f} s, "
              f"peak {self.peak_amplitude():.2f} mV")


# Grandchild: adds long-term (Holter) monitoring information
class HolterRecording(ECGRecording):
    def __init__(self, patient_id: str, sampling_rate_hz: float, samples: list,
                 lead: str = "II", hours: float = 24.0) -> None:
        super().__init__(patient_id, sampling_rate_hz, samples, lead)
        self.hours = hours

    def expected_samples(self) -> int:                            # new method
        return int(self.hours * 3600 * self.sampling_rate_hz)

    def describe(self) -> None:                                   # override
        print(f"[Holter, lead {self.lead}] {self.patient_id} - {self.hours:.0f} h at "
              f"{self.sampling_rate_hz:.0f} Hz = {self.expected_samples():,} samples expected")


ecg_samples = [0.1, 0.4, 1.2, 0.3, 0.0] * 50    # 250 samples = 1 second at 250 Hz

r1 = BiosignalRecording("P-001", 250, ecg_samples)
r2 = ECGRecording("P-002", 250, ecg_samples, lead="V1")
r3 = HolterRecording("P-003", 250, ecg_samples, hours=24)

r1.describe()
r2.describe()
r3.describe()

print(r2.duration_seconds())   # inherited method: ECGRecording never defined it
print(r3.peak_amplitude())     # inherited from ECGRecording


In [ ]:
# isinstance() reflects the full inheritance chain
print(isinstance(r3, HolterRecording))      # True
print(isinstance(r3, ECGRecording))         # True
print(isinstance(r3, BiosignalRecording))   # True
print(isinstance(r1, ECGRecording))         # False: a generic recording is not necessarily an ECG


### 3.4 Polymorphism

**Polymorphism** ("many forms") means different classes share a method name, and you can call it uniformly without caring which specific type you are dealing with.


In [ ]:
class EEGRecording(BiosignalRecording):
    def __init__(self, patient_id: str, sampling_rate_hz: float, samples: list, channels: int = 19) -> None:
        super().__init__(patient_id, sampling_rate_hz, samples)
        self.channels = channels

    def describe(self) -> None:
        print(f"[EEG, {self.channels} ch] {self.patient_id} - {self.duration_seconds():.1f} s "
              f"at {self.sampling_rate_hz:.0f} Hz")


recordings = [
    r1,
    r2,
    r3,
    EEGRecording("P-004", 500, [0.0, 0.1] * 250),
]

for rec in recordings:
    rec.describe()    # same call, different output per type


Python picks the right `describe()` based on the object's actual type at runtime, with no `if/elif` needed. This is called **dynamic dispatch**, and it is what lets frameworks call `.forward()` on your model without knowing which model you wrote.


## 4. Dunder Methods and Operator Overloading

**Dunder methods** (double-underscore, also called *magic methods*) let you define how your objects behave with built-in Python operations: `print()`, `+`, `len()`, `==`, `<`, `[]`, and so on.

You have already used one, `__init__`. There are many more.

We will build a `Signal` class, a natural fit for biomedical work: signals have a length (number of samples), can be added together (clean signal + noise), can be indexed, and can be compared by strength.


### 4.1 `__str__`: readable `print()` output

Without it, `print(s)` gives something useless like `<__main__.Signal object at 0x7f...>`.


In [ ]:
class Signal:
    def __init__(self, samples: list) -> None:
        self.samples = samples

    def __str__(self) -> str:    # called by print() and str()
        return f"Signal({self.samples})"


s = Signal([1, 2, 3])
print(s)       # Signal([1, 2, 3])
print(str(s))  # same


### 4.2 `__len__`: support for `len()`

`len(s)` = number of samples in the signal.


In [ ]:
class Signal:
    def __init__(self, samples: list) -> None:
        self.samples = samples

    def __str__(self) -> str:
        return f"Signal({self.samples})"

    def __len__(self) -> int:    # called by len()
        return len(self.samples)


s1 = Signal([1, 2, 3])
s2 = Signal([4, 5, 6, 7])

print(len(s1))  # 3 samples
print(len(s2))  # 4 samples


### 4.3 `__add__`: operator overloading with `+`

Adding two signals sample by sample is a perfect use of `__add__`: it is exactly how a clean signal plus noise gives a noisy signal.


In [ ]:
class Signal:
    def __init__(self, samples: list) -> None:
        self.samples = samples

    def __str__(self) -> str:
        return f"Signal({self.samples})"

    def __len__(self) -> int:
        return len(self.samples)

    def __add__(self, other: 'Signal') -> 'Signal':   # called by s1 + s2
        if len(self) != len(other):
            raise ValueError("Signals must have the same number of samples.")
        return Signal([a + b for a, b in zip(self.samples, other.samples)])


clean = Signal([1.0, 2.0, 3.0])
noise = Signal([0.1, -0.2, 0.05])

noisy = clean + noise     # calls clean.__add__(noise)
print(noisy)


### 4.4 `__getitem__`: index access with `[]`

This one is essential for PyTorch datasets (Section 6). Defining `__getitem__` lets you write `signal[i]`, slices like `signal[0:2]`, and even loop over the object with `for`.


In [ ]:
class Signal:
    def __init__(self, samples: list) -> None:
        self.samples = samples

    def __str__(self) -> str:
        return f"Signal({self.samples})"

    def __len__(self) -> int:
        return len(self.samples)

    def __add__(self, other: 'Signal') -> 'Signal':
        if len(self) != len(other):
            raise ValueError("Signals must have the same number of samples.")
        return Signal([a + b for a, b in zip(self.samples, other.samples)])

    def __getitem__(self, idx):    # called by signal[idx]
        return self.samples[idx]


s = Signal([10, 20, 30, 40])

print(s[0])      # 10
print(s[-1])     # 40
print(s[1:3])    # [20, 30]

for x in s:      # looping works too
    print(x, end=" ")
print()


### 4.5 `__eq__` and `__lt__`: comparison operators

We compare signals by their **RMS amplitude** (root-mean-square, a common measure of signal strength). Once `__lt__` is defined, `sorted()` just works on your objects. (Comparing by RMS is one design choice; you decide what "equal" and "less than" mean for your class.)


In [ ]:
class Signal:
    def __init__(self, samples: list) -> None:
        self.samples = samples

    def __str__(self) -> str:
        return f"Signal({self.samples})"

    def __len__(self) -> int:
        return len(self.samples)

    def __add__(self, other: 'Signal') -> 'Signal':
        if len(self) != len(other):
            raise ValueError("Signals must have the same number of samples.")
        return Signal([a + b for a, b in zip(self.samples, other.samples)])

    def __getitem__(self, idx):
        return self.samples[idx]

    def rms(self) -> float:
        return (sum(x**2 for x in self.samples) / len(self.samples)) ** 0.5

    def __eq__(self, other: 'Signal') -> bool:   # ==
        return self.rms() == other.rms()

    def __lt__(self, other: 'Signal') -> bool:   # <
        return self.rms() < other.rms()


s1 = Signal([1, 0, 0])
s2 = Signal([0, 1, 0])
s3 = Signal([3, 4, 0])

print(s1 == s2)    # True: both have the same RMS amplitude
print(s1 < s3)     # True: weaker signal < stronger signal

# Once __lt__ is defined, sorted() works on your objects!
for s in sorted([s3, s1, s2]):
    print(s, "- RMS:", round(s.rms(), 3))


### 4.6 `__call__`: objects that behave like functions

If a class defines `__call__`, you can use an instance like a function: `normalize(x)`. This is exactly how preprocessing steps (transforms) and neural networks are used in PyTorch.


In [ ]:
class ZScoreNormalizer:
    """Standardize a value: (x - mean) / std."""
    def __init__(self, mean: float, std: float) -> None:
        self.mean = mean
        self.std = std

    def __call__(self, x: float) -> float:     # called by normalize(x)
        return (x - self.mean) / self.std


normalize = ZScoreNormalizer(mean=70.0, std=10.0)

print(normalize(90.0))                          # 2.0 -> calls normalize.__call__(90.0)
print([normalize(hr) for hr in [60, 70, 80]])   # [-1.0, 0.0, 1.0]
print(callable(normalize))                      # True


### Common dunder methods: quick reference

| Method | Triggered by | Typical use |
|---|---|---|
| `__init__` | `MyClass(...)` | Constructor |
| `__str__` | `print(obj)`, `str(obj)` | Human-readable string |
| `__len__` | `len(obj)` | Size or dimensionality |
| `__add__` | `obj1 + obj2` | Addition / merging |
| `__eq__` | `obj1 == obj2` | Equality check |
| `__lt__` | `obj1 < obj2` | Less-than; enables `sorted()` |
| `__getitem__` | `obj[i]` | Index access |
| `__call__` | `obj(...)` | Make an instance callable like a function |

Keep `__getitem__`, `__len__`, and `__call__` in mind for Section 6: they are exactly the ones PyTorch relies on.


## 5. Class Attributes vs Instance Attributes

- **Instance attributes** belong to each object individually, defined via `self` in `__init__`.
- **Class attributes** are shared across all instances, defined at the class level.


In [ ]:
class Patient:
    hospital = "University Hospital"   # class attribute: same for every patient
    count = 0                          # class attribute: shared counter

    def __init__(self, name: str, age: int) -> None:
        self.name = name               # instance attribute: unique per object
        self.age = age
        Patient.count += 1


a1 = Patient("Ayse", 34)
a2 = Patient("Mehmet", 58)
a3 = Patient("Zeynep", 41)

print(a1.hospital)                          # University Hospital
print(Patient.hospital)                     # also reachable via the class name
print(f"Total patients registered: {Patient.count}")   # 3

print(a1.name, a2.name, a3.name)            # different per instance


## 6. Where OOP Comes Into Action for Deep Learning

Everything up to here was setup. Now we see where OOP actually pays off.

**PyTorch**, the most popular deep learning framework (you will learn it in depth later in the course), is built entirely around the OOP concepts from this notebook. You do not need to understand what these models *do* yet; the point right now is to **recognize the patterns**. Every idea below already appeared above.


### 6.1 A neural network is just a class

Here is a small PyTorch model that takes 8 clinical measurements of a patient and produces 2 class scores (for example "low risk" vs "high risk"). Read it as an OOP exercise, not an ML one, and spot the pieces you already know:

- It **inherits** from `nn.Module` (the parent class): `class ClinicalClassifier(nn.Module)`.
- Its constructor calls **`super().__init__()`**: the exact `super()` line we flagged in the Inheritance section.
- It stores layers as **instance attributes** via `self.` in `__init__`.
- It **overrides** a method, `forward()`, to define its own behavior: polymorphism, just like `describe()`.


In [ ]:
import torch
from torch import nn

class ClinicalClassifier(nn.Module):
    def __init__(self, n_features: int = 8, n_classes: int = 2):
        super().__init__()                       # inherit from nn.Module
        self.layers = nn.Sequential(             # layers stored as instance attributes
            nn.Linear(n_features, 16),
            nn.ReLU(),
            nn.Linear(16, n_classes),
        )

    def forward(self, x):                        # override: defines this model's behavior
        return self.layers(x)


model = ClinicalClassifier()
print(model)                 # nn.Module defines __str__ for you: readable output

x = torch.randn(4, 8)        # 4 patients x 8 clinical features (random numbers for now)
logits = model(x)            # you write model(x), NOT model.forward(x)
print(logits.shape)          # torch.Size([4, 2]): 2 class scores for each of the 4 patients


Notice that you write `model(x)`, not `model.forward(x)`. That works because `nn.Module` defines `__call__`, which internally calls your `forward()`. That is the `__call__` dunder from Section 4.6, doing real work in a real framework.


### 6.2 A dataset is just a class too

When you load data in PyTorch, you write a class that inherits from `Dataset` and implements two dunder methods you already met: **`__len__`** and **`__getitem__`**.

- `__len__` tells PyTorch how many samples exist: same dunder as `len(signal)`.
- `__getitem__` lets PyTorch grab sample `i` with `dataset[i]`: the index-access dunder from Section 4.4.

That is the entire contract. PyTorch's `DataLoader` then uses these two methods to batch and shuffle your data automatically, without knowing anything about your specific dataset (polymorphism again).


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ECGDataset(Dataset):
    def __init__(self, signals, labels):
        self.signals = signals       # instance attributes
        self.labels = labels

    def __len__(self):               # len(dataset) -> number of samples
        return len(self.labels)

    def __getitem__(self, idx):      # dataset[idx] -> one (signal, label) pair
        return self.signals[idx], self.labels[idx]


# Random numbers stand in for real ECG recordings:
# 20 recordings x 250 samples each, label 0 = normal, 1 = abnormal
torch.manual_seed(0)
signals = torch.randn(20, 250)
labels = torch.randint(0, 2, (20,))

dataset = ECGDataset(signals, labels)

print(len(dataset))                  # 20
x0, y0 = dataset[0]
print(x0.shape, y0)                  # one recording and its label

# DataLoader only needs __len__ and __getitem__ to build batches
loader = DataLoader(dataset, batch_size=4, shuffle=True)
for batch_signals, batch_labels in loader:
    print(batch_signals.shape, batch_labels.shape)   # torch.Size([4, 250]) torch.Size([4])
    break


Later in the course you will write a dataset class for real medical data, for example chest X-ray images listed in a CSV file. It follows the same pattern; only the body of `__getitem__` changes (it loads a file from disk instead of indexing a tensor). This cell is shown for reference and is not meant to be run:

```python
import os
import pandas as pd
from torch.utils.data import Dataset
from torchvision.io import decode_image

class ChestXrayDataset(Dataset):
    def __init__(self, annotations_csv, img_dir, transform=None):
        self.img_labels = pd.read_csv(annotations_csv)   # instance attributes
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)    # a transform is a callable object (__call__)
        return image, label
```


Look back at the classes above and match each line to a concept from this notebook:

| PyTorch code | OOP concept | Where you saw it |
|---|---|---|
| `class ClinicalClassifier(nn.Module)` | Inheritance | Section 3.3 |
| `super().__init__()` | Calling the parent constructor | Section 3.3 |
| `self.layers = ...` | Instance attributes | Sections 2, 5 |
| `def forward(self, x)` | Method override / polymorphism | Sections 3.3, 3.4 |
| `model(x)` calls `forward` | `__call__` dunder | Section 4.6 |
| `def __len__(self)` | `__len__` dunder | Section 4.2 |
| `def __getitem__(self, idx)` | `__getitem__` dunder | Section 4.4 |

None of this is new. PyTorch did not invent special syntax; it just leans all the way into Python's OOP. That is why this recitation comes before you touch the framework.


## Common beginner mistakes

- **Forgetting `self`** as the first parameter of a method: `TypeError: describe() takes 0 positional arguments but 1 was given`.
- **Forgetting `self.`** when storing or reading an attribute: the value stays a local variable and disappears after `__init__` finishes.
- **Forgetting `super().__init__()`** in a child class (especially in a PyTorch model): you will get errors such as `cannot assign module before Module.__init__() call`.
- **Calling `model.forward(x)`** instead of `model(x)`: it may appear to work, but it skips the machinery `nn.Module` adds in `__call__`.


## Wrap-Up

| Concept | Key idea |
|---|---|
| **Class / Object** | Blueprint and concrete instance |
| **`__init__` / `self`** | Constructor; reference to this specific object |
| **Encapsulation** | Bundle data + methods; control access with `_` / `__` |
| **Abstraction** | Hide internal complexity; expose a clean public interface |
| **Inheritance** | Child class reuses and extends parent via `super()` |
| **Polymorphism** | Same method name, different behavior per class |
| **Dunder methods** | Control how objects behave with `print`, `len`, `+`, `==`, `<`, `[]`, `()` |
| **Class attributes** | Shared across all instances (vs per-object instance attributes) |
| **PyTorch models & datasets** | Real classes built on every concept above |

**Next steps:**
- Run through the companion **Quick Check** notebook to self-test on these concepts.
- Rewrite the `Signal` class from a blank cell without looking. If you can, the dunder methods have landed.
- When PyTorch shows up later in the course, come back to Section 6. Every model and dataset you write will follow these exact patterns.

**Further reading:**
- [Python tutorial: Classes](https://docs.python.org/3/tutorial/classes.html)
- [Python data model: special method names](https://docs.python.org/3/reference/datamodel.html#special-method-names)
- [`abc`: Abstract Base Classes](https://docs.python.org/3/library/abc.html)
- [PyTorch: Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- [PyTorch: Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
